In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import hashlib
from datetime import datetime
import time

TEAM_ROLL_NUMBERS = ["25L-0012", "25L-2545"]  # <-- Apne roll numbers likhein
BASE_URL = "https://www.scrapethissite.com/pages/forms/"
RAW_CSV_FILENAME = "hockey_teams_raw.csv"

session = requests.Session()
session.trust_env = False 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
start_time = datetime.now()
all_rows = []
page_num = 1
requests_count = 0
print("Scraping started...")
while True:
    params = {"page_num": page_num, "per_page": 100}
    response = session.get(BASE_URL, params=params, headers=headers, timeout=15)
    requests_count += 1
    if response.status_code != 200:
        print(f"Failed to fetch page {page_num}. Status code: {response.status_code}")
        break
    soup = BeautifulSoup(response.text, "html.parser")
    table_rows = soup.find_all("tr", class_="team")
    if not table_rows:
        break
    for row in table_rows:
        record = {
            "Team Name": row.find("td", class_="name").get_text(strip=True),
            "Year": row.find("td", class_="year").get_text(strip=True),
            "Wins": row.find("td", class_="wins").get_text(strip=True),
            "Losses": row.find("td", class_="losses").get_text(strip=True),
            "OT Losses": row.find("td", class_="ot-losses").get_text(strip=True),
            "Win %": row.find("td", class_="pct").get_text(strip=True),
            "GF": row.find("td", class_="gf").get_text(strip=True),
            "GA": row.find("td", class_="ga").get_text(strip=True),
            "+/-": row.find("td", class_="diff").get_text(strip=True),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        all_rows.append(record)
    page_num += 1
    time.sleep(0.5)
end_time = datetime.now()
df_raw = pd.DataFrame(all_rows)
df_raw.to_csv(RAW_CSV_FILENAME, index=False)
sha256_hash = hashlib.sha256()
with open(RAW_CSV_FILENAME, "rb") as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)
file_fingerprint = sha256_hash.hexdigest()

print("=" * 60)
print("MANDATORY DATA COLLECTION LOG")
print("=" * 60)
print(f"Team Roll Numbers       : {', '.join(TEAM_ROLL_NUMBERS)}")
print(f"Source URL              : {BASE_URL}")
print(f"Collection Start Time   : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Collection End Time     : {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total Requests / Pages  : {requests_count}")
print(f"Total Rows Collected    : {len(df_raw)}")
print(f"Raw Dataset Filename    : {RAW_CSV_FILENAME}")
print(f"SHA-256 Fingerprint     : {file_fingerprint}")
print("=" * 60)

df_raw.head()

Scraping started...
MANDATORY DATA COLLECTION LOG
Team Roll Numbers       : 25L-0012, 25L-2545
Source URL              : https://www.scrapethissite.com/pages/forms/
Collection Start Time   : 2026-11-15 13:05:49
Collection End Time     : 2026-11-15 13:06:18
Total Requests / Pages  : 7
Total Rows Collected    : 582
Raw Dataset Filename    : hockey_teams_raw.csv
SHA-256 Fingerprint     : 1985bac902d455f29c40365373b8b44342819075c646e950bfa9b72e7e10cf85


,Team Name,Year,Wins,Losses,OT Losses,Win %,GF,GA,+/-,collected_at
0,Boston Bruins,1990,44,24,,0.55,299,264,35,2026-11-15 13:06:07
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14,2026-11-15 13:06:07
2,Calgary Flames,1990,46,26,,0.575,344,263,81,2026-11-15 13:06:07
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73,2026-11-15 13:06:07
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25,2026-11-15 13:06:07


1. This code is fetching the html table from the given website.
2. This is done inorder to collect live raw data and to prove data's authenticity 
3. It shows when the data is collected, how many rows are covered, total pages, and SHA-256 fingerprint